# 8. 循环神经网络 (Recurrent Neural Networks, RNN)

> 如果说卷积神经网络（CNN）是深度学习的“双眼”，擅长捕捉**空间**结构；
> 
> 那么循环神经网络（RNN）就是深度学习的“大脑”，擅长处理**时间**序列。

1. **核心矛盾：从“独立”到“关联”**

    - **之前的模型（MLP/CNN）**：假设样本之间是**独立同分布 (IID)** 的。输入一张猫，输出猫；输入下一张狗，与上一张无关。
    - **现实的序列**：在文本、语音、股市中，当前的数值极度依赖于过去。

      - 例子：“我吃了一个____”。（根据前文，这里大概率是“苹果”或“馒头”，而不是“飞机”）。

    - **RNN 的使命**：引入**隐藏状态 (Hidden State)**，让模型具备“记忆”能力。

2. **知识版图：本章的四个核心模块**
  
    - 第一模块：文本预处理 (Text Preprocessing)

        在把文字喂给神经网络前，必须经过洗练：
        1. **词元化 (Tokenization)**：把句子拆成单词或字符。
        2. **词表 (Vocabulary)**：建立“单词”到“数字 ID”的映射。
        3. **加载序列数据**：学习如何通过“随机采样”或“顺序分区”来切分长文本。

     - 第二模块：语言模型 (Language Models)
       
       1. **数学本质**：计算一个序列出现的概率 $P(x_1, x_2, \dots, x_T)$。
       2. **马尔可夫模型 (Markov Models)**：理解 $n$ 元语法 (n-grams) 的局限。
       3. **齐普夫定律 (Zipf's Law)**：理解真实语言中词频分布的极端不平衡（二八定律）。

     - 第三模块：循环神经网络架构 (RNN Architecture)
       
       1. **隐藏状态 ($H_t$)**：理解 $H_t = \phi(X_t W_{xh} + H_{t-1} W_{hh} + b_h)$。
       2. **循环特性**：理解为什么 $W_{hh}$（隐藏层到隐藏层的权重）是实现记忆的关键。
       3. **困惑度 (Perplexity)**：学习评价序列模型好坏的标准（不再仅仅是准确率）。

     - 第四模块：随时间反向传播 (BPTT)
       
       1. **核心痛点**：序列越长，计算图越深。
       2. **梯度危机**：深度理解为什么 RNN 天生容易产生**梯度消失**或**梯度爆炸**。
       3. **截断梯度**：学习工程上如何通过“梯度裁剪”来保证训练不崩溃。

---


## 8.1 序列模型 (Sequence Models)

### 1. 核心数学矛盾：维度的无穷增长

在之前的模型（如 MLP/CNN）中，我们处理的是固定维度的输入。但在序列数据中，为了预测 $t$ 时刻的输出 $x_t$，理论上我们需要考虑之前所有的观测值 $x_1, \dots, x_{t-1}$。

#### 1.1 概率链式法则 (Chain Rule)
任意序列的联合概率分布可以展开为：
$$P(x_1, \dots, x_T) = \prod_{t=1}^T P(x_t \mid x_1, \dots, x_{t-1})$$
- **工程挑战**：随着时间 $t$ 的推移，条件概率的输入向量 $x_1, \dots, x_{t-1}$ 的长度会不断增加，模型无法处理变长的、且趋于无穷大的特征向量。

---

### 2. 解决方案：简化假设

为了使计算可行，序列模型引入了两种核心策略：

#### 2.1 自回归模型 (Autoregressive Models)

假设当前值只取决于过去固定跨度 $\tau$ 内的观测值。
- **数学表达**：$P(x_t \mid x_{t-1}, \dots, x_1) \approx P(x_t \mid x_{t-1}, \dots, x_{t-\tau})$
- **术语**：这被称为 **$\tau$ 阶马尔可夫模型 (Markov Model)**。此时，特征的维度被固定为 $\tau$。

#### 2.2 潜变量模型 (Latent Variable Models)
不保留长长的历史，而是维护一个**隐藏状态 (Hidden State)** $h_t$ 来总结历史。
- **公式**：
    1.  $h_t = g(h_{t-1}, x_{t-1})$ （更新记忆）
    2.  $\hat{x}_t = P(x_t \mid h_t)$ （基于记忆预测）
- **意义**：这是 RNN 的理论基石。

---

### 3. 核心约束：因果性 (Causality)

**因果性**是序列建模的铁律：**预测 $x_t$ 时，模型只能接触到时间轴上 $t$ 以前的数据。**
- **体现**：在构造数据集时，标签必须是特征窗口之后的第一个点。
- **后果**：模型严禁“偷看未来”，任何违反此规则的设计都会导致训练出的模型在实际应用中毫无用处（数据泄露）。

---

### 4. 构造序列数据集

我们将通过预测带噪声的正弦波来演示如何构造符合因果性的训练集。


In [31]:
import torch
from torch import Tensor

class SequenceDataGenerator:
    """序列数据生成与处理器。
    
    负责生成模拟序列数据，并利用滑动窗口技术构造符合因果性约束的特征-标签对。
    """
    def __init__(self, n_points: int = 1000, tau: int = 4):
        """
        Args:
            n_points: 总数据点数。
            tau: 时间窗口大小（马尔可夫阶数）。
        """
        self.n_points = n_points
        self.tau = tau
        self.time = torch.arange(1, n_points + 1, dtype=torch.float32)
        # 生成带噪声的正弦波
        self.x = torch.sin(0.01 * self.time) + torch.normal(0, 0.2, (n_points,))

    def split_data(self, train_percent: float = 0.6) -> tuple[Tensor, Tensor, Tensor, Tensor]:
        """构造滑动窗口数据集并划分训练集与验证集。
        
        因果性体现：X[i] = [x[i], ..., x[i+tau-1]], y[i] = x[i+tau]。
        
        Returns:
            train_features: 形状为 (n_points - tau, tau) 的训练集特征矩阵。
            train_labels: 形状为 (n_points - tau, 1) 的训练集标签向量。
            val_features: 验证集特征矩阵。
            val_labels: 验证集标签向量。
        """
        num_samples = self.n_points - self.tau
        features = torch.zeros((num_samples, self.tau))
        
        # 填充特征矩阵：每一列都是序列的一个偏移版本
        for i in range(self.tau):
            features[:, i] = self.x[i : i + num_samples]
            
        # 标签是紧随窗口后的那个点，即从索引 tau 开始的所有点
        labels = self.x[self.tau:].reshape((-1, 1))
        
        # 按照 train_percent 划分数据集
        n_train = int(num_samples * train_percent)
        train_features = features[:n_train]
        train_labels = labels[:n_train]
        val_features = features[n_train:]
        val_labels = labels[n_train:]
        
        return train_features, train_labels, val_features, val_labels

In [32]:
# --- 验证数据构造 ---
gen = SequenceDataGenerator(tau=4)
train_features, train_labels, _, _ = gen.split_data()
print(f"特征示例 (前4个点): {train_features[0]}")
print(f"标签示例 (第5个点): {train_labels[0]}")

特征示例 (前4个点): tensor([-0.3304, -0.0782,  0.3960,  0.2389])

标签示例 (第5个点): tensor([0.0574])


---

### 5. 多步预测 (Multistep Prediction) 的崩溃

这是 8.1 节最重要的结论，揭示了简单自回归模型的局限性。

#### 5.1 单步预测 (One-step-ahead)

-  **定义**：始终使用**真实**的历史数据预测下一个点。
-  **表现**：由于输入是准确的，模型表现通常非常优秀。

#### 5.2 多步预测 (Multistep / K-step-ahead)

-  **定义**：使用模型**自己生成的预测值**作为输入，继续预测更远的未来。
-  **表现**：**误差累积 (Error Accumulation)**。
-  **数学直觉**：如果单步预测有误差 $\epsilon$，在 $k$ 步预测中，误差会按指数级传播。你会发现预测曲线迅速偏离真实轨迹。

---

### 6. 总结 Checklist

-  **自回归策略**：理解如何将变长序列转化为固定长度的特征向量。
-  **因果性约束**：掌握滑动窗口切片时“不准看未来”的索引逻辑。
-  **马尔可夫阶数 ($\tau$)**：明白窗口大小决定了模型“回看”历史的深度。
-  **误差累积**：深刻理解为什么简单的自回归模型无法进行长程预测（多步预测会崩溃）。

---


## 8.2 文本预处理 (Text Preprocessing)

> 在 NLP（自然语言处理）中，我们常面对的是非结构化的文本。
> 
> 神经网络无法直接处理“单词”，因此我们需要一套标准化的流程，将原始文本转化为模型可计算的数字索引。
>
> 本节我们将实现一个完整的**文本预处理流水线**，涵盖读取、词元化、词表构建和索引转换。

### 1. 核心工作流 (The Pipeline)

一个标准的文本预处理流程包含四个步骤：
1. **读取文本**：将原始文件加载到内存。
2. **词元化 (Tokenization)**：将字符串拆分为词元（Tokens）。
3. **构建词表 (Vocabulary)**：建立词元到数字 ID 的映射。
4. **语料转换 (Corpus Mapping)**：将文本序列转换为数字索引序列。

---

### 2. 构建预处理组件

我们将使用 H.G. Wells 的《时光机器》（The Time Machine）作为实验语料。

#### 2.1 读取与初步清洗


In [33]:
import re
import requests
import hashlib
from pathlib import Path
from typing import Union

def download_time_machine(data_dir: Union[str, Path] = 'temp/data/plaintext') -> Path:
    """幂等下载《时光机器》数据集。
    
    Args:
        data_dir: 数据存储目录。
        
    Returns:
        Path: 下载后的文件 Path 对象。
        
    Raises:
        RuntimeError: 下载失败或网络异常时抛出。
    """
    url = 'http://d2l-data.s3-accelerate.amazonaws.com/timemachine.txt'
    expected_sha1 = '090b5e7e70c295757f55df93cb0a180b9691891a'
    
    data_path = Path(data_dir)
    data_path.mkdir(parents=True, exist_ok=True)
    file_path = data_path / 'timemachine.txt'
    
    # 幂等校验逻辑
    if file_path.exists():
        content = file_path.read_bytes()
        if hashlib.sha1(content).hexdigest() == expected_sha1:
            print(f"Dataset already exists and verified: {file_path}")
            return file_path
    
    # 执行下载
    print(f"Downloading dataset from {url}...")
    try:
        response = requests.get(url, stream=True, timeout=10)
        response.raise_for_status()
        
        # 写入文件并验证
        file_path.write_bytes(response.content)
        
        # 下载后的二次校验
        if hashlib.sha1(file_path.read_bytes()).hexdigest() != expected_sha1:
            file_path.unlink(missing_ok=True)
            raise RuntimeError("Data integrity check failed after download.")
            
        print(f"Download complete: {file_path}")
        
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Failed to download dataset: {e}")
        
    return file_path

def read_time_machine() -> list[str]:
    """读取《时光机器》数据集并逐行进行标准化清洗。

    Returns:
        list[str]: 包含清洗后文本行的列表。
    """
    file_path = download_time_machine()
    
    # 使用 utf-8 编码读取并处理
    with file_path.open('r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # 清洗逻辑：正则过滤、去除冗余空格、统一转小写
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

In [34]:
lines = read_time_machine()
print(f'# 文本总行数: {len(lines)}')
print(lines[0])
print(lines[10])

Dataset already exists and verified: temp/data/plaintext/timemachine.txt

# 文本总行数: 3221

the time machine by h g wells

twinkled and his usually pale face was flushed and animated the

#### 2.2 词元化 (Tokenization)

词元是文本处理的最小单位。


In [35]:
def tokenize(lines: list[str], token_type: str = 'word') -> list[list[str]]:
    """将文本行拆分为词元。

    Args:
        lines: 文本行列表。
        token_type: 词元类型，'word'（单词）或 'char'（字符）。

    Returns:
        list[list[str]]: 每一行对应的词元列表。
    """
    if token_type == 'word':
        return [line.split() for line in lines]
    elif token_type == 'char':
        return [list(line) for line in lines]
    else:
        raise ValueError(f"错误：不支持的词元类型 {token_type}")
    
tokens = tokenize(lines,)
for i in range(11):
    print(tokens[i])

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']

[]

[]

[]

[]

...

[]

[]

['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']

['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']

['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']

#### 2.3 词表类 (The Vocab Class)

这是本节最核心的工程组件。它负责维护 `Token <-> Index` 的映射。


In [36]:
import collections
from typing import Any, Optional, Counter as CounterType

class Vocab:
    """文本词表类。

    负责统计词频、过滤低频词，并提供词元与索引的双向映射。

    Attributes:
        token_to_idx (dict[str, int]): 词元到索引的映射。
        idx_to_token (list[str]): 索引到词元的映射。
    """
    def __init__(
        self, 
        tokens: Optional[list[Union[str, list[str]]]] = None, 
        min_freq: int = 0, 
        reserved_tokens: Optional[list[str]] = None
    ) -> None:
        """初始化词表。

        Args:
            tokens: 词元列表或嵌套列表。
            min_freq: 最小词频阈值，低于此值的词元将被标记为 <unk>。
            reserved_tokens: 预留的特殊词元（如 <pad>, <bos>, <eos>）。
        """
        if tokens is None: tokens = []
        if reserved_tokens is None: reserved_tokens = []

        # 1. 统计词频 (并降序排序)
        flatten_tokens = self._flatten(tokens)
        counter: CounterType[str] = collections.Counter(flatten_tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        self.freq_dict = dict(counter)

        # 2. 初始映射（<unk> 的索引永远是 0）
        self.idx_to_token: list[str] = ['<unk>'] + reserved_tokens
        self.token_to_idx: dict[str, int] = {token: i for i, token in enumerate(self.idx_to_token)}

        # 3. 构建索引
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self) -> int:
        """返回词表大小。"""
        return len(self.idx_to_token)

    def __getitem__(self, tokens: Union[str, list[str], tuple[str, ...]]) -> Union[int, list[int]]:
        """根据词元获取索引。"""
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.token_to_idx['<unk>'])
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices: Union[int, list[int]]) -> Union[str, list[str]]:
        """根据索引获取词元。"""
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    def _flatten(self, tokens: Any) -> list[str]:
        """递归展平嵌套的词元列表。"""
        result = []
        for item in tokens:
            if isinstance(item, list):
                result.extend(self._flatten(item))
            else:
                result.append(item)
        return result

    @property
    def unk(self) -> int:
        """返回未知词元的索引。"""
        return 0
    
    @property
    def token_freqs(self):
        """返回词频。"""
        return self._token_freqs
    
    def frequency(self, token: str) -> int:
        """查询特定词的出现次数。"""
        return self.freq_dict.get(token, 0)

In [37]:
from rich import print
vocab = Vocab(tokens)
print(list(vocab.token_to_idx.items())[:10])

[
    ('<unk>', 0),
    ('the', 1),
    ('i', 2),
    ('and', 3),
    ('of', 4),
    ('a', 5),
    ('to', 6),
    ('was', 7),
    ('in', 8),
    ('that', 9)
]


---

### 3. 整合：语料库加载器


In [38]:
def load_corpus_time_machine(
    max_tokens: int = -1, 
    token_type: str = 'char'
) -> tuple[list[int], Vocab]:
    """加载《时光机器》语料库并转换为索引。

    Args:
        max_tokens: 截断的最大长度 (若非正，则全部)。
        token_type: 'char' 或 'word'。

    Returns:
        corpus: 索引化的长序列。
        vocab: 构建好的词表。
    """
    lines = read_time_machine()
    tokens = tokenize(lines, token_type)
    vocab = Vocab(tokens)
    
    # 展平所有行，形成一个单一的长序列
    corpus = [vocab[token] for line in tokens for token in line]
    
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab

# --- 验证程序 ---
corpus, vocab = load_corpus_time_machine(token_type='word')
print(f"词表大小: {len(vocab)}")
print(f"语料库前10个索引: {corpus[:10]}")
print(f"还原后的文本: {' '.join(vocab.to_tokens(corpus[:10]))}")

Dataset already exists and verified: temp/data/plaintext/timemachine.txt

词表大小: 4580

语料库前10个索引: [1, 19, 50, 40, 2183, 2184, 400, 2, 1, 19]

还原后的文本: the time machine by h g wells i the time


---

### 4. 细致梳理

1. **词元化策略对比**：
    - **单词级 (Word)**：语义明确，但词表很大（通常几万个），且容易遇到未知词（OOV）。
    - **字符级 (Char)**：词表极小（几十到几百个），没有 OOV 问题，但单个词元语义缺失，序列长度大幅增加。
2.  **特殊词元 (Special Tokens)**：
    - `<unk>`：Unknown，代表词表中未出现的词。
    - `<pad>`：Padding，用于填充句子到固定长度。
    - `<bos>` / `<eos>`：句子的开始和结束。
3.  **词频过滤 (Min Frequency)**：
    - **工程意义**：训练数据中只出现一两次的词（长尾词）通常是噪声或拼写错误，过滤它们能显著减小模型参数量并提高泛化能力。

---

### 5. 关键工程暗知识 (Engineering Wisdom)

*   **`Counter` 的效率**：在处理百万级语料时，`collections.Counter` 是 Python 中统计词频的最快方式。
*   **内存优化**：在 `Vocab` 中使用 `token_to_idx`（字典）和 `idx_to_token`（列表）是空间与时间效率的最优组合，支持 $O(1)$ 级别的双向查找。
*   **正则化清洗**：在 8.2 节中，我们只保留字母。在实际工程（如处理中文）中，还需要考虑分词（jieba）、去除停用词等步骤。

---

### 6. 总结 Checklist

*   **Tokenization**：理解词与字符的区别。
*   **Vocab 映射**：掌握如何将文本数字化。
*   **特殊词元**：明白 `<unk>` 的必要性。

---
